# Belief analysis

Another outcome analysis alongside `outcome_analysis.ipynb` and `efficiency_analysis.ipynb`, this time on **stated belief** (`collabBelief`, 0-100) -- each partner's own reported belief, before deciding, that their partner will act collaboratively that round.

Unlike the categorical outcome or `E`, `collabBelief` is an **individual** measure, not a joint one -- each of the two partners reports their own belief independently. So this reshapes `task_data.csv` to one row per (participant, round) rather than one row per round, with `arm` and the round's task-difficulty measures (shared by both partners, since they depend on both of that round's tasks) carried along.

**Scope**: this notebook covers what predicts `collabBelief` itself, and what a participant's *own* stated belief predicts about their *own* subsequent choice. The parallel question -- whether a participant's *partner's* belief predicts their choice -- is split out into `partner_belief_analysis.ipynb`, since it depends on this notebook's models as controls but asks a genuinely distinct question (about the belief-sharing mechanism specifically, rather than belief as an individual measure).

**A caveat that matters for interpreting any `arm` effect found here.** Per `public/index.js`, `collabBelief` is submitted *before* the robot recommendation becomes accessible each round -- the round starts with the belief slider enabled and the robot button disabled; the robot only unlocks after both partners have submitted their belief for that round. So the robot cannot be mechanically shaping *that round's* stated belief -- there's no direct within-round causal path from "consulted the robot" to "reported this belief." Any `arm` effect on `collabBelief` has to be explained some other way; two candidates, not mutually exclusive:

1. **A knowledge/incentive effect.** The robot's info modal explicitly displays `"Partner Belief = " + partnerCollabBelief` -- so a participant's stated belief is shown to their partner (via the robot) once both have submitted. Treatment participants who've experienced this may report more optimistic beliefs as a strategic or self-presentational signal, rather than a private honest forecast -- a measurement validity concern specific to the treatment arm, since control participants have no such sharing mechanism at all.
2. **A contextual/cumulative effect.** Repeated exposure to the treatment condition and its occasional recommendations across the 30 rounds could shift general trust or optimism over the course of the session, showing up in every round's stated belief regardless of whether the robot was consulted *that* round.

Nothing in this dataset can cleanly separate these two (or rule out some mix of both) -- keep that in mind throughout.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

task = pd.read_csv("task_data.csv")
task_summary = pd.read_csv("task_summary.csv")
task.head()

,arm,session,round,username_1,username_2,task_1,task_2,design_1,design_2,strategy_1,strategy_2,collabBelief_1,collabBelief_2,usedRobot_1,usedRobot_2,score_1,score_2
0,control,1,1,user0011,user0012,6,1,M,M,C,C,72,79,False,False,122.0,122.0
1,control,1,2,user0011,user0012,0,13,Y,L,I,C,83,60,False,False,50.0,-85.0
2,control,1,3,user0011,user0012,20,2,Y,L,I,C,50,70,False,False,50.0,-28.0
3,control,1,4,user0011,user0012,15,4,K,K,C,C,100,75,False,False,105.0,100.0
4,control,1,5,user0011,user0012,3,25,L,Y,C,I,0,60,False,False,11.0,50.0


## Drop missing data and add task difficulty

Same criterion as the other two outcome notebooks, kept for consistency across the three analytic samples (even though `collabBelief` is actually still present and valid for these 2 rounds -- only `design`/`strategy`/`score` failed to register).

In [2]:
missing = (task["strategy_1"] == "undefined") | (task["strategy_2"] == "undefined")
print(f"Dropping {missing.sum()} of {len(task)} rounds with an undefined strategy.")
task = task[~missing].copy()
task["pair_id"] = task["username_1"] + "_" + task["username_2"]

real_tasks = task_summary[task_summary["task_difficulty"].notna()].copy()
difficulty_by_index = real_tasks.set_index("task_index")["task_difficulty"]
magnitude_by_index = real_tasks.set_index("task_index")["payoff_magnitude"]
difficulty_1 = task["task_1"].map(difficulty_by_index).astype(int)
difficulty_2 = task["task_2"].map(difficulty_by_index).astype(int)
task["max_difficulty"] = np.maximum(difficulty_1, difficulty_2)
task["diff_difficulty"] = (difficulty_1 - difficulty_2).abs()

Dropping 2 of 780 rounds with an undefined strategy.


## Reshape to one row per (participant, round)

In [3]:
shared_cols = ["arm", "pair_id", "round", "max_difficulty", "diff_difficulty"]

belief = pd.concat([
    task[shared_cols + ["username_1", "task_1", "collabBelief_1", "collabBelief_2", "strategy_1"]].rename(
        columns={"username_1": "username", "task_1": "task", "collabBelief_1": "collabBelief",
                 "collabBelief_2": "partner_belief", "strategy_1": "strategy"}),
    task[shared_cols + ["username_2", "task_2", "collabBelief_2", "collabBelief_1", "strategy_2"]].rename(
        columns={"username_2": "username", "task_2": "task", "collabBelief_2": "collabBelief",
                 "collabBelief_1": "partner_belief", "strategy_2": "strategy"}),
], ignore_index=True)

belief["chose_C"] = (belief["strategy"] == "C").astype(int)
belief["own_magnitude"] = belief["task"].map(magnitude_by_index)
belief["collabBelief_c"] = belief["collabBelief"] - belief["collabBelief"].mean()
belief["partner_belief_c"] = belief["partner_belief"] - belief["partner_belief"].mean()
belief["max_difficulty_c"] = belief["max_difficulty"] - belief["max_difficulty"].mean()
belief["diff_difficulty_c"] = belief["diff_difficulty"] - belief["diff_difficulty"].mean()
belief["own_magnitude_c"] = belief["own_magnitude"] - belief["own_magnitude"].mean()

print(f"n={len(belief)} (participant, round) observations, {belief['username'].nunique()} participants, "
      f"{belief['pair_id'].nunique()} pairs")
belief["collabBelief"].describe()

n=1556 (participant, round) observations, 52 participants, 26 pairs


count    1556.000000
mean       76.038560
std        28.136846
min         0.000000
25%        63.000000
50%        87.000000
75%        99.000000
max       100.000000
Name: collabBelief, dtype: float64

## `collabBelief` by arm

In [4]:
belief_by_arm = belief.groupby("arm")["collabBelief"].agg(["count", "mean", "std", "min", "median", "max"])
belief_by_arm.loc["overall"] = belief["collabBelief"].agg(["count", "mean", "std", "min", "median", "max"])
belief_by_arm.round(2)

,count,mean,std,min,median,max
arm,,,,,,
control,716.0,65.38,30.36,0.0,73.0,100.0
treatment,840.0,85.12,22.42,0.0,96.0,100.0
overall,1556.0,76.04,28.14,0.0,87.0,100.0


## Inferential statistics: pair-level aggregation

Same non-independence issue as the other two notebooks: each pair collapsed to its own mean `collabBelief` across both partners and all rounds, giving one independent observation per pair (12 control, 14 treatment).

In [5]:
pair_summary = belief.groupby(["arm", "pair_id"])["collabBelief"].mean().reset_index()

control_vals = pair_summary.loc[pair_summary["arm"] == "control", "collabBelief"]
treatment_vals = pair_summary.loc[pair_summary["arm"] == "treatment", "collabBelief"]


def cohens_d(a, b):
    n_a, n_b = len(a), len(b)
    pooled_sd = (((n_a - 1) * a.var(ddof=1) + (n_b - 1) * b.var(ddof=1)) / (n_a + n_b - 2)) ** 0.5
    return (a.mean() - b.mean()) / pooled_sd


t_stat, t_p = stats.ttest_ind(treatment_vals, control_vals, equal_var=False)
u_stat, u_p = stats.mannwhitneyu(treatment_vals, control_vals, alternative="two-sided")

print(f"control:   n={len(control_vals)}, mean={control_vals.mean():.2f}, sd={control_vals.std():.2f}")
print(f"treatment: n={len(treatment_vals)}, mean={treatment_vals.mean():.2f}, sd={treatment_vals.std():.2f}")
print(f"Cohen's d = {cohens_d(treatment_vals, control_vals):.3f}")
print(f"Welch t = {t_stat:.3f}, p = {t_p:.4f}")
print(f"Mann-Whitney U = {u_stat:.1f}, p = {u_p:.4f}")

control:   n=12, mean=65.32, sd=18.43
treatment: n=14, mean=85.12, sd=12.33
Cohen's d = 1.283
Welch t = 3.163, p = 0.0052
Mann-Whitney U = 135.0, p = 0.0094


## Basic model: `collabBelief ~ arm`

A linear mixed model (`MixedLM`, REML) with a random intercept per pair, cross-checked against GEE (Gaussian family, cluster-robust SEs on `pair_id`) -- the same pairing used in `efficiency_analysis.ipynb`, which doesn't share the variational-Bayes anti-conservatism `outcome_analysis.ipynb` had to work around for its binary outcome.

In [6]:
mixed_basic = smf.mixedlm("collabBelief ~ arm", data=belief, groups=belief["pair_id"]).fit()
print(mixed_basic.summary())

           Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: collabBelief
No. Observations:   1556    Method:             REML        
No. Groups:         26      Scale:              482.6074    
Min. group size:    58      Log-Likelihood:     -7053.5493  
Max. group size:    60      Converged:          Yes         
Mean group size:    59.8                                    
------------------------------------------------------------
                  Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept         65.327    4.455 14.664 0.000 56.595 74.058
arm[T.treatment]  19.795    6.071  3.261 0.001  7.896 31.693
Group Var        230.060    3.154                           



In [7]:
gee_basic = smf.gee("collabBelief ~ arm", groups="pair_id", data=belief, family=sm.families.Gaussian()).fit()
print(gee_basic.summary())

                               GEE Regression Results                              
Dep. Variable:                collabBelief   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Gaussian   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                         695.281
Covariance type:                    robust   Time:                         11:59:37
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           65.3827      5.106     12.805      0.000      55.375  

Unlike every other `arm` effect in this analysis, this one is clearly significant -- at every level (pair-level test above, and both round-level models here), and `MixedLM` and GEE agree closely (coef ~19.8, p = 0.001 in both). Treatment participants report substantially higher belief that their partner will collaborate -- but see the caveat above: since belief is reported before the robot is even accessible that round, this isn't evidence the robot directly shapes that round's stated belief.

## Adding task difficulty

Same derived covariates as the other two notebooks: `max_difficulty_c` (the harder of the two partners' tasks that round) and `diff_difficulty_c` (how mismatched their difficulties are).

In [8]:
difficulty_formula = "collabBelief ~ arm + max_difficulty_c + diff_difficulty_c"

mixed_difficulty = smf.mixedlm(difficulty_formula, data=belief, groups=belief["pair_id"]).fit()
print(mixed_difficulty.summary())

            Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  collabBelief
No. Observations:  1556     Method:              REML        
No. Groups:        26       Scale:               470.2986    
Min. group size:   58       Log-Likelihood:      -7032.4572  
Max. group size:   60       Converged:           Yes         
Mean group size:   59.8                                      
-------------------------------------------------------------
                   Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------
Intercept          65.313    4.458 14.650 0.000 56.575 74.050
arm[T.treatment]   19.820    6.075  3.263 0.001  7.913 31.727
max_difficulty_c   -3.278    0.510 -6.430 0.000 -4.277 -2.279
diff_difficulty_c   1.290    0.510  2.529 0.011  0.290  2.290
Group Var         230.613    3.199                           



In [9]:
gee_difficulty = smf.gee(
    difficulty_formula, groups="pair_id", data=belief, family=sm.families.Gaussian(),
).fit()
print(gee_difficulty.summary())

                               GEE Regression Results                              
Dep. Variable:                collabBelief   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Gaussian   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                         683.753
Covariance type:                    robust   Time:                         11:59:37
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            65.3695      5.113     12.784      0.000      55.34

## Interpretation

- **`arm` is significant and essentially unchanged by adding difficulty** (coef ~19.8, p = 0.001 in both models, before and after) -- treatment participants report substantially more belief that their partner will collaborate, independent of task difficulty. This is the clearest, most robust `arm` effect anywhere in this analysis, at every level (pair-level, round-level, both estimation methods).
- **`max_difficulty_c` is negative and significant** (p = 0.002 in both): harder tasks (with harsher downside payoffs) reduce stated belief in the partner's cooperation, a sensible result -- riskier situations breed more caution about trusting a partner.
- **`diff_difficulty_c` is positive and significant** (p = 0.02-0.03 in both), a smaller effect: belief rises slightly as the partners' task difficulties diverge. This wasn't hypothesized going in and doesn't have an obvious story yet -- worth treating as a finding to investigate further rather than over-interpreting here. An `arm:diff_difficulty_c` interaction (as tested for the other two outcomes) would be a natural next step, given the pattern found there.

**On what the `arm` effect means**, given the timing caveat from the intro: this is *not* evidence that "the tool changes what people believe," in the sense of the robot's recommendation directly informing that round's stated belief -- the belief is submitted before the robot is even unlockable that round. What it does show, robustly, is that treatment participants report higher `collabBelief` than control participants throughout the session. Two explanations seem most plausible, and this data can't distinguish them: (1) since the robot modal reveals a participant's stated belief to their partner, treatment participants may be reporting a strategic/self-presentational number rather than a private honest forecast, in a way control participants have no reason to; or (2) cumulative exposure to the treatment condition over the 30 rounds shifts general trust or optimism, independent of any single round's tool use.

Read alongside `outcome_analysis.ipynb` and `efficiency_analysis.ipynb`, the honest summary is: the treatment arm reports much higher stated belief in the partner throughout, and separately shows some behavioral trends (buffering against difficulty mismatch) that only reach significance in some specifications. Whether the belief measure is a real psychological mediator of those behavioral patterns, or largely a reporting/incentive artifact of the belief-sharing mechanism, isn't something this dataset can settle -- it should be presented as a description of what was reported, not as a validated explanation for the behavioral results.

## Does belief predict the participant's own strategy choice?

A different question from everything above: not what predicts *stated belief*, but whether a participant's own belief that round predicts their own subsequent strategy choice (`C`/`I`), controlling for `arm` and task difficulty. Unlike the robot-timing issue in the intro, this direction is temporally valid -- per `public/index.js`, the design table is blocked (`if (!$("#collabBelief").prop("disabled")) return`) until `collabBelief` is submitted, so belief is reported *before* the design/strategy choice, every round.

`chose_C ~ collabBelief_c + arm + max_difficulty_c + diff_difficulty_c`, on the same reshaped (participant, round) data as above. `chose_C` is binary, so this uses `BinomialBayesMixedGLM` again, with the same caveat as `outcome_analysis.ipynb`: trust GEE over the mixed model's SE for `arm` (a pair-level covariate), but `collabBelief_c` and the difficulty terms vary within pair and within person round-to-round, so both models should agree on those.

In [10]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

strategy_formula = "chose_C ~ collabBelief_c + arm + max_difficulty_c + diff_difficulty_c"

gee_strategy = smf.gee(strategy_formula, groups="pair_id", data=belief, family=sm.families.Binomial()).fit()
print(gee_strategy.summary())

                               GEE Regression Results                              
Dep. Variable:                     chose_C   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Binomial   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:59:37
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             1.6931      0.358      4.727      0.000       0.99

In [11]:
mixed_strategy = BinomialBayesMixedGLM.from_formula(
    strategy_formula, vc_formulas={"pair": "0 + C(pair_id)"}, data=belief,
).fit_vb()
print(mixed_strategy.summary())

                   Binomial Mixed GLM Results
                  Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
----------------------------------------------------------------
Intercept            M     2.2816   0.0897                      
arm[T.treatment]     M     1.7622   0.1484                      
collabBelief_c       M     0.0401   0.0030                      
max_difficulty_c     M    -0.7145   0.0795                      
diff_difficulty_c    M     0.0887   0.0706                      
pair                 V     1.0050   0.1348 2.732   2.086   3.578
Parameter types are mean structure (M) and variance structure
(V)
Variance parameters are modeled as log standard deviations


### Interpretation

- **`collabBelief_c` is a strong, robust predictor of the participant's own strategy choice** (GEE coef ≈ 0.040, p < 0.001; mixed model coef ≈ 0.040, p < 0.001 -- the two agree closely, as expected for a within-person, within-pair varying covariate). A participant who states a higher belief that their partner will collaborate is substantially more likely to choose `C` themselves that same round.
- **`max_difficulty_c` is negative and significant in both** (harder tasks reduce the odds of choosing `C`), consistent with every other difficulty result in this analysis.
- **`diff_difficulty_c` is not significant** (GEE p ≈ 0.67).
- **`arm` is not significant in GEE** (p ≈ 0.96) once `collabBelief` and difficulty are controlled -- the mixed model's much larger, seemingly significant `arm` coefficient here is the same pair-level-covariate artifact documented in `outcome_analysis.ipynb`; GEE is the one to trust. In other words, `arm` has no *additional* association with strategy choice beyond what's already captured by a participant's own stated belief and the task's difficulty -- consistent with belief (whatever is driving it) sitting causally upstream of the choice, at least in this cross-sectional sense.

This doesn't resolve the measurement question from the intro -- whether `collabBelief` is a private forecast or partly a performative number -- but it does show that whatever participants report, they tend to act consistently with it: stated belief and same-round choice line up strongly and robustly, independent of `arm` and task difficulty.

### Does the treatment change any of these relationships?

Before settling on the model above, all three possible `arm` interactions were tested (`collabBelief_c:arm`, `max_difficulty_c:arm`, `diff_difficulty_c:arm`) in a single model. Only one was real: `collabBelief_c:arm` was a weak, inconsistent trend (GEE p = 0.076, but the mixed model showed almost no effect at all -- 0.004 vs GEE's 0.025), and `max_difficulty_c:arm` wasn't close to significant in either model (the two didn't even agree on its sign). `diff_difficulty_c:arm` was significant and consistent in both (GEE p = 0.016, mixed model coef 0.57 with a clear z), so it's added here on its own, dropping the other two for a more parsimonious model -- the same approach used for the categorical outcome in `outcome_analysis.ipynb`.

In [12]:
interaction_formula = (
    "chose_C ~ collabBelief_c + arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c"
)

gee_interaction = smf.gee(interaction_formula, groups="pair_id", data=belief, family=sm.families.Binomial()).fit()
print(gee_interaction.summary())

                               GEE Regression Results                              
Dep. Variable:                     chose_C   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Binomial   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:59:38
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                             

In [13]:
mixed_interaction = BinomialBayesMixedGLM.from_formula(
    interaction_formula, vc_formulas={"pair": "0 + C(pair_id)"}, data=belief,
).fit_vb()
print(mixed_interaction.summary())

                           Binomial Mixed GLM Results
                                   Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------------------------------
Intercept                             M     2.3330   0.0904                      
arm[T.treatment]                      M     1.7363   0.1482                      
collabBelief_c                        M     0.0412   0.0030                      
max_difficulty_c                      M    -0.7365   0.0801                      
diff_difficulty_c                     M    -0.0815   0.0712                      
arm[T.treatment]:diff_difficulty_c    M     0.5021   0.1201                      
pair                                  V     1.0170   0.1348 2.765   2.111   3.620
Parameter types are mean structure (M) and variance structure (V)
Variance parameters are modeled as log standard deviations


### Interpretation

`collabBelief_c` (coef ≈ 0.040, p < 0.001) and `max_difficulty_c` (coef ≈ -0.47, p < 0.001) are essentially unchanged from the model without the interaction. `arm` alone remains non-significant in GEE (p = 0.886) -- still no *uniform* effect of `arm` on strategy choice.

**`arm:diff_difficulty_c` is significant in both models** (GEE p = 0.020, mixed model corroborates with the same sign and a larger, clearly significant coefficient). In control, `diff_difficulty_c`'s own coefficient is negative (-0.125, not significant alone, p = 0.294) -- a mild pull away from `C` as the partners' task difficulties diverge. In treatment, that slope shifts by +0.338, flipping the effective slope positive. This is the third independent confirmation of the same pattern found for the categorical outcome (`outcome_analysis.ipynb`) and for efficiency (`efficiency_analysis.ipynb`): the treatment specifically offsets the harm of partner-difficulty mismatch, here visible even at the level of individual strategy choice after controlling for the person's own stated belief -- so this isn't just an artifact of belief differing by arm, it holds up as an independent effect on behavior.

### Extending the model: does payoff magnitude matter too?

A different kind of factor from everything above: `task_summary.csv`'s `payoff_magnitude` (1-5) scales a task's payoffs up or down while holding its risk ratio `u` nearly fixed within a `task_difficulty` tier (see `risk_dominance_analysis.ipynb`) -- so two tasks at the same difficulty but different magnitude present the *same relative gamble* at different absolute stakes. A natural risk-aversion hypothesis: participants might be more willing to collaborate when the absolute stakes are small, even at a *relatively* risky task, and more cautious at the same relative risk once the stakes are scaled up.

This adds `own_magnitude_c` -- the participant's own task's `payoff_magnitude`, mean-centered -- to the established model. All three possible new terms are tested together first: the main effect, `max_difficulty_c:own_magnitude_c` (does the magnitude penalty compound with difficulty), and `arm:own_magnitude_c` (does the treatment change it).

In [14]:
magnitude_test_formula = (
    "chose_C ~ collabBelief_c + arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c "
    "+ own_magnitude_c + max_difficulty_c:own_magnitude_c + arm:own_magnitude_c"
)

gee_magnitude_test = smf.gee(magnitude_test_formula, groups="pair_id", data=belief, family=sm.families.Binomial()).fit()
print(gee_magnitude_test.summary())

                               GEE Regression Results                              
Dep. Variable:                     chose_C   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Binomial   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:59:39
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                             

`own_magnitude_c` and `max_difficulty_c:own_magnitude_c` are both significant; `arm:own_magnitude_c` is not (p = 0.326) -- dropped for a more parsimonious model, the same approach used throughout this analysis.

In [15]:
magnitude_formula = (
    "chose_C ~ collabBelief_c + arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c "
    "+ own_magnitude_c + max_difficulty_c:own_magnitude_c"
)

gee_magnitude = smf.gee(magnitude_formula, groups="pair_id", data=belief, family=sm.families.Binomial()).fit()
print(gee_magnitude.summary())

                               GEE Regression Results                              
Dep. Variable:                     chose_C   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Binomial   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:59:39
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                             

In [16]:
mixed_magnitude = BinomialBayesMixedGLM.from_formula(
    magnitude_formula, vc_formulas={"pair": "0 + C(pair_id)"}, data=belief,
).fit_vb()
print(mixed_magnitude.summary())

                           Binomial Mixed GLM Results
                                   Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------------------------------
Intercept                             M     2.3720   0.0917                      
arm[T.treatment]                      M     1.7758   0.1501                      
collabBelief_c                        M     0.0417   0.0031                      
max_difficulty_c                      M    -0.7381   0.0819                      
diff_difficulty_c                     M    -0.0506   0.0724                      
arm[T.treatment]:diff_difficulty_c    M     0.5165   0.1216                      
own_magnitude_c                       M    -0.1947   0.0649                      
max_difficulty_c:own_magnitude_c      M    -0.2025   0.0696                      
pair                                  V     1.0446   0.1347 2.842   2.171   3.721
Parameter types are mean structure (M) and v

### Interpretation

`collabBelief_c` (coef ≈ 0.040, p < 0.001), `max_difficulty_c` (coef ≈ -0.46, p < 0.001), and `arm:diff_difficulty_c` (GEE p = 0.019, mixed model corroborates) are all essentially unchanged from the model without magnitude -- adding it doesn't disturb any of the established results.

- **`own_magnitude_c` is significant and negative** (GEE coef -0.104, p = 0.001; mixed model corroborates in sign and magnitude): holding relative risk (`task_difficulty`, and therefore `u`) fixed, a participant facing a task with bigger absolute stakes is less likely to choose `C`. Since `payoff_magnitude` scales a task's payoffs while holding its risk *ratio* nearly constant, this can't be explained by the gamble becoming relatively worse -- only bigger. This is the risk-aversion signature the magnitude factor was designed to detect.
- **`max_difficulty_c:own_magnitude_c` is significant and negative** (GEE p = 0.002; mixed model corroborates): the magnitude penalty is not constant, it compounds with difficulty. At an easy task, bigger stakes barely matter; at a hard task, they matter substantially more -- consistent with a risk-aversion effect that's amplified precisely when the underlying gamble is already risky, rather than a uniform aversion to "big numbers" regardless of context.
- **`arm:own_magnitude_c` is not significant** (tested, not shown -- p = 0.326): the treatment doesn't change how magnitude relates to choice. Unlike the well-established `arm:diff_difficulty_c` buffering effect, this looks like an individual psychological response to stake size, not something the robot or belief-sharing mechanism touches.

Taken together: task difficulty, magnitude, and their interaction all shape a participant's own strategy choice independently of (and in addition to) their own stated belief -- `collabBelief` doesn't fully mediate any of these effects, since all three survive controlling for it essentially unchanged.

## Sensitivity check: risk dominance (`u`/`R`) instead of task difficulty

`outcome_analysis_selected_u.ipynb` replaced `max_difficulty`/`diff_difficulty` with covariates built from the risk threshold `u` of the *specific collaborative design each participant actually selected* -- `u`, unlike `task_difficulty`, is computed from real per-task payoffs (correcting the task-19 mislabeling from `data/README.md` automatically) and reflects which design was chosen, not just which task was assigned. That analysis also found `max_u_c` collinear enough with `diff_u_c` (r ≈ 0.35) to distort a mechanistic claim, and got a cleaner result using `R` (the average log-odds risk threshold, r ≈ -0.17 with `diff_u_c`) instead of `max_u_c`.

This repeats that substitution here: `R_c` and `diff_u_c` (both round-shared, exactly as `max_difficulty_c`/`diff_difficulty_c` were) in place of `max_difficulty_c`/`diff_difficulty_c` in the belief-predicts-strategy model established above, to see whether the `arm:diff_difficulty_c` buffering interaction is a property of the specific task-difficulty measure or holds up under this alternative, more precise risk measure too.

In [17]:
import json

with open("../data/experiment.json", encoding="utf-8") as f:
    experiment = json.load(f)

COLLAB_LABELS = ["Design K", "Design L", "Design M"]
rank_by_task = {}
for task_index in range(30):
    options_by_label = {option["label"]: option for option in experiment["tasks"][task_index]["options"]}
    ranked = sorted(COLLAB_LABELS, key=lambda label: int(options_by_label[label]["upside"]), reverse=True)
    rank_by_task[task_index] = {
        label.replace("Design ", ""): rank_letter
        for rank_letter, label in zip(["A", "B", "C"], ranked)
    }


def compute_u(design):
    v_ci = real_tasks[f"V_{design}_CI"]
    v_cc = real_tasks[f"V_{design}_CC"]
    return (real_tasks["V_Y_II"] - v_ci) / ((real_tasks["V_Y_II"] - v_ci) + (v_cc - real_tasks["V_Y_IC"]))


real_tasks["u_A"] = compute_u("A")
real_tasks["u_B"] = compute_u("B")
real_tasks["u_C"] = compute_u("C")
u_by_task_and_rank = real_tasks.set_index("task_index")[["u_A", "u_B", "u_C"]].to_dict("index")


def u_selected(task_index, design):
    if design == "Y":
        return u_by_task_and_rank[task_index]["u_A"]
    rank_letter = rank_by_task[task_index][design]
    return u_by_task_and_rank[task_index][f"u_{rank_letter}"]


task["u_selected_1"] = task.apply(lambda r: u_selected(r["task_1"], r["design_1"]), axis=1)
task["u_selected_2"] = task.apply(lambda r: u_selected(r["task_2"], r["design_2"]), axis=1)
task["diff_u"] = (task["u_selected_1"] - task["u_selected_2"]).abs()


def logit(p):
    return np.log(p / (1 - p))


task["R"] = 0.5 * logit(task["u_selected_1"]) + 0.5 * logit(task["u_selected_2"])

belief = belief.merge(task[["pair_id", "round", "diff_u", "R"]], on=["pair_id", "round"], how="left")
belief["diff_u_c"] = belief["diff_u"] - belief["diff_u"].mean()
belief["R_c"] = belief["R"] - belief["R"].mean()

print("correlation(R_c, diff_u_c):", belief["R_c"].corr(belief["diff_u_c"]))

correlation(R_c, diff_u_c): -0.1679943946201091


Both possible `arm` interactions are tested together before settling on a final model, the same approach used throughout this analysis -- and `own_magnitude_c` (with its interactions) is carried over from the `max_difficulty_c`-based section above, to check whether that finding also holds under this alternative risk measure.

In [18]:
u_interaction_formula = (
    "chose_C ~ collabBelief_c + arm + R_c + diff_u_c + arm:diff_u_c "
    "+ own_magnitude_c + R_c:own_magnitude_c + arm:own_magnitude_c"
)

gee_u_interaction = smf.gee(u_interaction_formula, groups="pair_id", data=belief, family=sm.families.Binomial()).fit()
print(gee_u_interaction.summary())

                               GEE Regression Results                              
Dep. Variable:                     chose_C   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Binomial   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:59:39
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            1.716

`arm:own_magnitude_c` is not significant (p = 0.371) and `R_c:own_magnitude_c` falls just short (p = 0.098) -- unlike `max_difficulty_c:own_magnitude_c` in the earlier section (p = 0.002), the magnitude-compounds-with-risk interaction doesn't clearly replicate under this measure. Both are dropped, keeping `own_magnitude_c` as a main effect alongside the already-established `arm:diff_u_c`.

In [19]:
u_formula = "chose_C ~ collabBelief_c + arm + R_c + diff_u_c + arm:diff_u_c + own_magnitude_c"

gee_u = smf.gee(u_formula, groups="pair_id", data=belief, family=sm.families.Binomial()).fit()
print(gee_u.summary())

                               GEE Regression Results                              
Dep. Variable:                     chose_C   No. Observations:                 1556
Model:                                 GEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                           Binomial   Mean cluster size:                59.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Fri, 04 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:59:39
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     1.7262      0.363      4.7

In [20]:
mixed_u = BinomialBayesMixedGLM.from_formula(
    u_formula, vc_formulas={"pair": "0 + C(pair_id)"}, data=belief,
).fit_vb()
print(mixed_u.summary())

                       Binomial Mixed GLM Results
                          Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
------------------------------------------------------------------------
Intercept                    M     2.2833   0.0897                      
arm[T.treatment]             M     1.7733   0.1480                      
collabBelief_c               M     0.0399   0.0030                      
R_c                          M    -2.3349   0.2755                      
diff_u_c                     M    -3.6866   0.9640                      
arm[T.treatment]:diff_u_c    M     2.7736   1.3624                      
own_magnitude_c              M    -0.2001   0.0634                      
pair                         V     1.0133   0.1348 2.755   2.104   3.607
Parameter types are mean structure (M) and variance structure (V)
Variance parameters are modeled as log standard deviations


### Interpretation

The buffering interaction replicates under this alternative risk measure, and `own_magnitude_c` survives alongside it:

- **`collabBelief_c` is essentially unchanged** (GEE coef 0.040, p < 0.001; mixed model agrees) -- the belief-predicts-choice result doesn't depend on how difficulty is measured.
- **`R_c` is significant and negative** (GEE coef -1.61, p < 0.001; mixed model corroborates in sign, larger magnitude as usual for this artifact): higher overall risk reduces the odds of choosing `C`, the same role `max_difficulty_c` played.
- **`diff_u_c` (the control-arm slope) is significant and negative** (GEE coef -5.54, p = 0.005) -- here, unlike with `diff_difficulty_c` (which wasn't significant alone, p = 0.67), mismatch on its own has a clear negative association with choosing `C` in control.
- **`arm:diff_u_c` is significant** (GEE coef 5.41, p = 0.019; mixed model agrees in sign). The effective treatment-arm slope (roughly -5.54 + 5.41 ≈ -0.13) is close to flat, the same buffering shape found with `diff_difficulty_c` -- control's mismatch penalty is substantially offset in treatment.
- **`own_magnitude_c` is significant and negative** (GEE coef -0.118, p < 0.001; mixed model coef -0.200, corroborates in sign): bigger absolute stakes reduce the odds of choosing `C`, holding overall risk fixed -- the same risk-aversion effect found in the `max_difficulty_c`-based section, replicating under this alternative risk measure too.
- **`arm` alone remains non-significant** (GEE p = 0.911), the same conclusion as every specification in this analysis.

One difference from the earlier section: the `max_difficulty_c:own_magnitude_c` compounding interaction (magnitude mattering more at harder tasks) doesn't clearly replicate here -- `R_c:own_magnitude_c` falls just short of significance (p = 0.098). The magnitude *main effect* is robust across both risk measures; the *compounding-with-difficulty* refinement is specific to the `max_difficulty_c` version and should be read as more tentative.

This is now a fourth independent confirmation of the buffering interaction specifically at the level of individual strategy choice (after the original `task_difficulty`-based version here, and the categorical/efficiency outcomes), and it holds up under a covariate built from actual selected-design payoffs rather than assigned task difficulty -- with the added benefit that `diff_u_c`'s own main effect is cleanly significant here, unlike the more ambiguous `diff_difficulty_c` version.